In [1]:
# ResearchAssignment3 - Hack Day Prep
# Laine Kowalski
# ASTR 400B Final Project


#RA2 Feedback from Gurtina/More simplified outline:
"""
*focusing on too many things - mass profile, rotation 
curve, velocity dispersion, angular momentum. You need to narrow 
down more. 

Study the kinematics of the merged halo vs what the MW halo and 
M31 halo before the merger:

1)	pick two points in time, one before they start to merge 
(e.g. snapshot 0) and one after the merger 
(note that you identified when this merger happened already in 
Homework 6). 

2)	compute the average angular momentum of 
the MW halo/M31 halo/merged halo at a few different radii 
(you could make a profile, where you compute this as a function of 
radius). How would you define the merged MW + M31 halo? 
How would you compute angular momentum? 

3)	Compare the three results.  - Does the merger remnant have the 
same profile as either of the initial galaxies? Is the profile 
smooth?  If it is different, why might that have happened 
(recalling that the mass or velocity could be the culprit)? 
"""

'\n*focusing on too many things - mass profile, rotation \ncurve, velocity dispersion, angular momentum. You need to narrow \ndown more. \n\nStudy the kinematics of the merged halo vs what the MW halo and \nM31 halo before the merger:\n\n1)\tpick two points in time, one before they start to merge \n(e.g. snapshot 0) and one after the merger \n(note that you identified when this merger happened already in \nHomework 6). \n\n2)\tcompute the average angular momentum of \nthe MW halo/M31 halo/merged halo at a few different radii \n(you could make a profile, where you compute this as a function of \nradius). How would you define the merged MW + M31 halo? \nHow would you compute angular momentum? \n\n3)\tCompare the three results.  - Does the merger remnant have the \nsame profile as either of the initial galaxies? Is the profile \nsmooth?  If it is different, why might that have happened \n(recalling that the mass or velocity could be the culprit)? \n'

In [2]:
#Goal of Code
"""
1. At the beginning of your code write a comment that states the 
topic of your research project.

2. Some of you have outlined multiple questions to pursue. 
For this assignment pick ONE of those questions and add it to the 
comments.

3. Many of you have outlined multiple ideas of plots to create or 
calculations to make. For this assignment pick ONE of those 
ideas and add it to the comments.

4. The title of the code should be informative with regards to 
what the code does.

"""

'\n1. At the beginning of your code write a comment that states the \ntopic of your research project.\n\n2. Some of you have outlined multiple questions to pursue. \nFor this assignment pick ONE of those questions and add it to the \ncomments.\n\n3. Many of you have outlined multiple ideas of plots to create or \ncalculations to make. For this assignment pick ONE of those \nideas and add it to the comments.\n\n4. The title of the code should be informative with regards to \nwhat the code does.\n\n'

In [3]:
#First code attempt:

"""
Milkdromeda Dark Halo vs MW and M31 Individual Profiles:

This code will investigate the the kinematics of the merged halo vs 
what the MW halo and M31 halo kinematics were before the merger. The
code picks two points in time, one before the galaxies begin to merge 
(Snapshot 0) and one after the merger (Snapshot ~610-630ish). Then,
the code will compute the average angular momentum of MW, M31 and the
merged Milkdromeda dark matter halo at different radii. 
The code will then compare the three results to see if the 
profiles are similar/different and the reason why will be explored.
"""

"""
All objects with a bin radii (everything within some delta), I can find
all angular momentum there and find the average. I do this for
many bins to get a histogram Ang Mom vs Radii profile.
"""

"""
OUTLINE:
# Import functions and libraries
# Read in of the simulation files
# Origin of the coordinate system is arbitrary at some location 
#Compute COM's of MW and M31 individually first
#Compute angular momentum of MW and M31 dark matter halo particles individually
#Find COM of Milkdromeda at around more than 6.7 Gyrs (Snapshot 600+) - still have to code this
#Compute angular momentum of halo particles wrt COM position of each particle as a function of the radius
and averaged them within radius shells
    #L = m*(r_CM x v_CM) 
#Making plots of MW, M31 and Milkdromeda angular momentum as a function
of radii and then compare them
#Hypothesis - total angular momentum of the merged system should 
be conserved
"""

"\nOUTLINE:\n# Import functions and libraries\n# Read in of the simulation files\n# Origin of the coordinate system is arbitrary at some location \n#Compute COM's of MW and M31 individually first\n#Compute angular momentum of MW and M31 dark matter halo particles individually\n#Find COM of Milkdromeda at around more than 6.7 Gyrs (Snapshot 600+)\n#Compute angular momentum of halo particles wrt COM position of each\nparticle\n    #L = m*(r_CM x v_CM) \n#Making plots of MW, M31 and Milkdromeda angular momentum as a function\nof radii and then compare them\n#Hypothesis - total angular momentum of the merged system should \nbe conserved\n"

In [4]:
#BEGIN CODE:
# Importing modules
import numpy as np
import astropy.units as u
from astropy.constants import G

# Import plotting modules
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

# Importing HW modules
from ReadFile import Read

# Modified CenterOfMass so that COM_P now takes a parameter specifying 
#by how much to decrease RMAX instead of a factor of 2
from CenterOfMass2 import CenterOfMass

In [44]:
#Creating COM Objects for MW and M31
#By default we are always starting at Snapshot 0

def ComputeAngMom(MergerSnapshot, delta_r):
    """
    This function computes the profile of the average angular momentum for the MW, M31 premerger and the combined
    system post merger.
    
    PARAMETERS:
        MergerSnapshot:
        delta_r:
    
    OUTPUTS:
    """
    
    #PRE-MERGER MW:
    #Defining COM objects pre-merger for MW and M31 at Snapshot 0
    COM_MW_0 = CenterOfMass(filename="MW_000.txt", ptype=1)
    
    #Getting the center of mass coordinates for both individual galaxies at Snapshot 0 (pre-merger)
    #Delta is like a tolerance - within 0.1 kpc, and volDec is the factor we are decreasing the volume until we get convergence of COM location
    r_COM_MW_0 = COM_MW_0.COM_P(delta=0.1,volDec=2).value #used similar values from the homeworks - shrinking down until COM stops changing
    
    #Storing the velocity vector of the MW COM (.value to get rid of units)
    v_COM_MW_0 = COM_MW_0.COM_V(r_COM_MW_0[0]*u.kpc, r_COM_MW_0[1]*u.kpc, r_COM_MW_0[2]*u.kpc).value
    
    #Setting up loop for MW halo particles pre-merger
    #Reading in data in the given file using Read function at Snapshot 0
    time, total, data_MW = Read("MW_000.txt")
    #create an index object for ptype=1 (Halo particles)                               
    index = np.where(data_MW['type'] == 1) #find rows where ptype=1 in data
    data_MW = data_MW[index] #limiting to particles whose ptype=1 in the array
    
    num_particles = len(data_MW) #initial number of halo particles for MW, gives number of rows, which is number of particles
    r_lower = 0 #initializing the lower radius
    r_upper = delta_r #will later add dr to both to increase it
    AngMomMW = [] #set up array to store ang mom
    particle_nums = [] #testing bc don't see 50000 particles
    
    #looping over all shells of upper and lower radius, but need a condition to tell us to stop maybe when no halo particles?
    while num_particles > 0:
        #print(num_particles)
        print(r_upper)
        L, num_particles = Ave_L(data_MW, r_COM_MW_0, v_COM_MW_0, r_lower, r_upper)
        AngMomMW.append(L) #calls and return Ave_L function, returning L's for that radius
        
        r_lower = r_lower + delta_r
        r_upper = r_upper + delta_r #Iterating towards the next step
        particle_nums.append(num_particles)
    
    
    
    
    
    #PRE-MERGER M31:
    #Defining COM objects pre-merger for MW and M31 at Snapshot 0
    COM_M31_0 = CenterOfMass(filename="M31_000.txt", ptype=1)
    
    #Getting the center of mass coordinates for both individual galaxies at Snapshot 0 (pre-merger)
    #Delta is like a tolerance - within 0.1 kpc, and volDec is the factor we are decreasing the volume until we get convergence of COM location
    r_COM_M31_0 = COM_M31_0.COM_P(delta=0.1,volDec = 4).value #used similar values from the homeworks - shrinking down until COM stops changing
    
    #Storing the velocity vector of the M31 COM (.value to get rid of units)
    v_COM_M31_0 = COM_M31_0.COM_V(r_COM_M31_0[0]*u.kpc, r_COM_M31_0[1]*u.kpc, r_COM_M31_0[2]*u.kpc).value
    
    
   

    #POST MERGER
        
    return AngMomMW, particle_nums
    

In [49]:
def Ave_L(data_in, r_COM, v_COM, r_lower, r_upper):
    """
    This function finds all of the particles in between r1 and r2. Then, for all of these particles, the 
    function will compute a new position and a new velocity relative to the COMs position and velocity 
    defined. Then, the function will compute the new angular momentgum wrt to the new COM.
    
    PARAMETERS:
        data: "array" - Array that contains all simulation data
        r_COM: "vector" - position vector of the COM of some galaxy
        v_COM: "vector" - velocity vector of the COM of some galaxy
        r_lower: "float" - lower radius of the bin under consideration
        r_upper: "float" - upper radius of the bin under consideration
        
    OUTPUTS:
        Lx_ave: x-comp of the average angular momentum within a bin
        Ly_ave: y-comp of the average angular momentum within a bin
        Lz_ave: z-comp of the average angular momentum within a bin
    """
    data = np.copy(data_in) #make a copy of object, modify copy without changing object
    
    # store the mass, positions, velocities of only the particles of the given type
    m = data['m'] #Storing the mass
    #Refering to all positions/velocities (x,y,z) for ptype=1
    x = data['x']
    y = data['y']
    z = data['z']
    vx = data['vx']
    vy = data['vy']
    vz = data['vz']
    
    #Computing relative position magnitude (distance) from the COM
    r = np.sqrt((x-r_COM[0])**2 + (y-r_COM[1])**2 + (z-r_COM[2])**2)
    
    #Creating a bin index to get the particles within the delta/bin that we want 
    #Finding rows in data files that match these conditions
    condition1 = r >= r_lower
    condition2 = r <= r_upper
    comb_conditions = np.logical_and(condition1, condition2)
    bin_index = np.where(comb_conditions)
 
    #Redefine positions and velocities to be the new positions and velocities wrt to 
    #the COM, so subtract COM off
    m = m[bin_index] #grabbing mass of individual particles in a bin
    x = x[bin_index] - r_COM[0]
    y = y[bin_index] - r_COM[1]
    z = z[bin_index] - r_COM[2]
    vx = vx[bin_index] - v_COM[0]
    vy = vy[bin_index] - v_COM[1]
    vz = vz[bin_index] - v_COM[2]
    num_particles = len(m)
    
    #Compute the average angular momentum of each individual particle within a bin
    #Multiplies by m and adds them all up, then divides by # of particles (we don't know how many in bin yet)
    #Cross product and then it is also an array of these numbers for the particles, dotting won't work with shape
    #So,I picked out one mass and multiplied every number in a row by the mass, summed them and then divide by the particle # to get the average
    #Boolean statement ensures we do not get an error dividing by 0 when there is no mass left to loop over
    if num_particles > 0:
        L = m[0]*np.sum(np.cross([x,y,z], [vx,vy,vz], axisa=0, axisb=0), axis=0) / num_particles
    else:
        L = np.zeros((3)) #getting a shape of [0,0,0] for Lx,Ly,Lz - set ang mom to 0 once no particles left
 
    return L, num_particles
#for upper and lower r's return the ave ang mom.

In [50]:
#Call function to get MW halo particles before merger

COM_MW_0 = CenterOfMass(filename="MW_000.txt", ptype=1)
r_COM_MW_0 = COM_MW_0.COM_P(0.1,2).value 

#Storing the velocity vector of the MW COM (.value to get rid of units)
v_COM_MW_0 = COM_MW_0.COM_V(r_COM_MW_0[0]*u.kpc, r_COM_MW_0[1]*u.kpc, r_COM_MW_0[2]*u.kpc).value

#Reading in data in the given file using Read function
time, total, data_MW = Read("MW_000.txt")
#create an index object for ptype=1 (Halo particles)                               
index = np.where(data_MW['type'] == 1)
data_MW = data_MW[index] #limiting to only halo particles

#Inputting values - assuming MW diam=30kpc (radius=15kpc) - TESTING ARBITRARY SHELL
Ave_L(data_MW, r_COM=r_COM_MW_0, v_COM=v_COM_MW_0, r_lower=10, r_upper=10.5)

#there are 991 particles in this shell

(array([-0.12981302,  0.09632723,  0.269254  ]), 85)

In [51]:
#Testing who function - weird results - bug somewhere
AngMomMW, particle_nums = ComputeAngMom(MergerSnapshot=None, delta_r=0.5)
#Dark matter halo shouldn't be concentrated at the center

0.5
1.0
1.5
2.0
2.5
3.0
3.5
4.0
4.5
5.0
5.5
6.0
6.5
7.0
7.5
8.0
8.5
9.0
9.5
10.0
10.5
11.0
11.5
12.0
12.5
13.0
13.5
14.0
14.5
15.0
15.5
16.0
16.5
17.0
17.5
18.0
18.5
19.0
19.5
20.0
20.5
21.0
21.5
22.0
22.5
23.0
23.5
24.0
24.5
25.0
25.5
26.0
26.5
27.0
27.5
28.0
28.5
29.0
29.5
30.0
30.5
31.0
31.5
32.0
32.5
33.0
33.5
34.0
34.5
35.0
35.5
36.0
36.5
37.0
37.5
38.0
38.5
39.0
39.5
40.0
40.5
41.0
41.5
42.0
42.5
43.0
43.5
44.0
44.5
45.0
45.5
46.0
46.5
47.0
47.5
48.0
48.5
49.0
49.5
50.0
50.5
51.0
51.5
52.0
52.5
53.0
53.5
54.0
54.5
55.0
55.5
56.0
56.5
57.0
57.5
58.0
58.5
59.0
59.5
60.0
60.5
61.0
61.5
62.0
62.5
63.0
63.5
64.0
64.5
65.0
65.5
66.0
66.5
67.0
67.5
68.0
68.5
69.0
69.5
70.0
70.5
71.0
71.5
72.0
72.5
73.0
73.5
74.0
74.5
75.0
75.5
76.0
76.5
77.0
77.5
78.0
78.5
79.0
79.5
80.0
80.5
81.0
81.5
82.0
82.5
83.0
83.5
84.0
84.5
85.0
85.5
86.0
86.5
87.0
87.5
88.0
88.5
89.0
89.5
90.0
90.5
91.0
91.5
92.0
92.5
93.0
93.5
94.0
94.5
95.0
95.5
96.0
96.5
97.0
97.5
98.0
98.5
99.0
99.5
100.0
100.5
101.0
101.5


In [53]:
sum(particle_nums) 
#something is wrong here because there should be around 50000 particles and we are going to over 600 kpc 
#- which seems too far from the COM

42408